# Notebook 7: Baseline Suite

## 왜 이 노트북이 필요한가?

현재 유일한 베이스라인은 "PCA 없이 원본 임베딩으로 IF" — trivially 질 straw man이다.
리뷰어가 가장 먼저 요구하는 것은 의미 있는 다수 베이스라인과의 비교다.

5개 베이스라인을 구현하고 합성 벤치마크에서 제안 방법과 비교한다:
- B0: Random (AUC=0.5 sanity check)
- B1: Position-only (덱 길이 편차)
- B2: CLIP 임베딩 + Global IF (학습 없음)
- B3: DINOv2 ViT-S + Global IF (학습 없음)
- B4: SBERT (텍스트 없으면 skip)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q open-clip-torch sentence-transformers timm
print('설치 완료')

## 0. 공통 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import ast, json, pickle
from pathlib import Path
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 합성 벤치마크 로드
benchmark_df = pd.read_csv(f'{LABELS_DIR}/synthetic_anomaly_benchmark.csv')
benchmark_df['sequence'] = benchmark_df['sequence'].apply(ast.literal_eval)
benchmark_labels = benchmark_df['is_anomaly'].values
print(f'벤치마크: {len(benchmark_df)}개 (정상={sum(benchmark_labels==0)}, 이상={sum(benchmark_labels==1)})')

# 학습 데이터 통계 (position baseline용)
df_train = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')
deck_lengths = df_train.groupby('deck_id').size()
length_mean = float(deck_lengths.mean())
length_std  = float(deck_lengths.std())
print(f'정상 덱 길이 평균: {length_mean:.1f} ± {length_std:.1f}')

## B0: Random Baseline

In [ ]:
np.random.seed(42)
random_scores = np.random.rand(len(benchmark_labels))
auc_b0 = roc_auc_score(benchmark_labels, random_scores)
print(f'B0 Random AUC: {auc_b0:.4f}  (expected ≈ 0.50)')

## B1: Position-only Baseline

덱 길이가 정상 분포에서 얼마나 벗어나는지로만 이상을 탐지.

In [ ]:
def length_anomaly_score(seq_len: int) -> float:
    z = abs(seq_len - length_mean) / (length_std + 1e-8)
    return float(np.clip(z / 3.0, 0, 1))

position_scores = np.array([
    length_anomaly_score(len(row['sequence']))
    for _, row in benchmark_df.iterrows()
])
auc_b1 = roc_auc_score(benchmark_labels, position_scores)
print(f'B1 Position-only AUC: {auc_b1:.4f}')
if auc_b1 > 0.7:
    print('⚠ position만으로도 잘 탐지됨 — 합성 이상이 길이 변화를 동반함')

## B2: CLIP 임베딩 + Global IF

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(device).eval()

class SimpleImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths     = [p for p in paths if Path(p).exists()]
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert('RGB')), self.paths[idx]

def extract_clip_emb(image_paths, batch_size=64):
    ds     = SimpleImageDataset(image_paths, clip_preprocess)
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
    all_emb, all_paths = [], []
    with torch.no_grad():
        for imgs, paths in loader:
            emb = clip_model.encode_image(imgs.to(device))
            emb = emb / emb.norm(dim=-1, keepdim=True)
            all_emb.append(emb.cpu().numpy())
            all_paths.extend(paths)
    return np.concatenate(all_emb), all_paths

# 학습 데이터 CLIP 임베딩
print('학습 데이터 CLIP 임베딩 추출 중...')
train_paths = df_train['image_path'].tolist()
clip_train_emb, _ = extract_clip_emb(train_paths)

from sklearn.ensemble import IsolationForest
clip_iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
clip_iso.fit(clip_train_emb)
print(f'CLIP IF 학습 완료: {clip_train_emb.shape}')

In [ ]:
# 벤치마크 슬라이드 CLIP 임베딩 추출 및 스코어링
# 벤치마크는 시퀀스 기반이므로 시퀀스 내 이미지 경로를 df_train에서 조회
bench_scores_clip = []
for _, row in benchmark_df.iterrows():
    orig_id = row['original_deck_id']
    deck_imgs = df_train[df_train['deck_id'] == orig_id].sort_values('slide_idx')['image_path'].tolist()
    if not deck_imgs:
        bench_scores_clip.append(0.5); continue
    emb, _ = extract_clip_emb(deck_imgs[:len(row['sequence'])])
    raw = clip_iso.decision_function(emb)
    s_min, s_max = raw.min(), raw.max()
    scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)
    bench_scores_clip.append(float(scores.mean()))

auc_b2 = roc_auc_score(benchmark_labels, bench_scores_clip)
print(f'B2 CLIP+IF AUC: {auc_b2:.4f}')

## B3: DINOv2 ViT-S + Global IF

In [ ]:
import timm

dino_model = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0)
dino_model = dino_model.to(device).eval()
dino_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def extract_dino_emb(image_paths, batch_size=64):
    ds     = SimpleImageDataset(image_paths, dino_transform)
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
    all_emb = []
    with torch.no_grad():
        for imgs, _ in loader:
            emb = dino_model(imgs.to(device))
            all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb)

print('학습 데이터 DINOv2 임베딩 추출 중...')
dino_train_emb = extract_dino_emb(train_paths)
dino_iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
dino_iso.fit(dino_train_emb)
print(f'DINOv2 IF 학습 완료: {dino_train_emb.shape}')

bench_scores_dino = []
for _, row in benchmark_df.iterrows():
    orig_id = row['original_deck_id']
    deck_imgs = df_train[df_train['deck_id'] == orig_id].sort_values('slide_idx')['image_path'].tolist()
    if not deck_imgs:
        bench_scores_dino.append(0.5); continue
    emb = extract_dino_emb(deck_imgs[:len(row['sequence'])])
    raw = dino_iso.decision_function(emb)
    s_min, s_max = raw.min(), raw.max()
    scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)
    bench_scores_dino.append(float(scores.mean()))

auc_b3 = roc_auc_score(benchmark_labels, bench_scores_dino)
print(f'B3 DINOv2+IF AUC: {auc_b3:.4f}')

## B4: SBERT (텍스트 없으면 skip)

In [ ]:
text_col = 'text' if 'text' in df_train.columns else None
if text_col is None:
    auc_b4 = None
    print('B4 SBERT skip — stanford_slide는 이미지 전용 데이터셋 (텍스트 필드 없음)')
else:
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer('all-MiniLM-L6-v2')
    texts = df_train[text_col].fillna('').tolist()
    sbert_emb = sbert.encode(texts, batch_size=256, show_progress_bar=True)
    sbert_iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
    sbert_iso.fit(sbert_emb)
    # 벤치마크 스코어링 ...
    print(f'B4 SBERT+IF AUC: {auc_b4:.4f}')

## 제안 방법 스코어 로드 및 전체 비교

In [ ]:
# 제안 방법 (NB05에서 계산된 HMM 스코어 재사용)
with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)

proposed_hmm_scores = []
for _, row in benchmark_df.iterrows():
    seq = np.array(row['sequence']).reshape(-1, 1)
    if len(seq) < 2:
        proposed_hmm_scores.append(0.5); continue
    ll = hmm_model.score(seq) / len(seq)
    z  = (thresholds['mean'] - ll) / (thresholds['std'] + 1e-8)
    proposed_hmm_scores.append(float(np.clip(z / 3.0, 0, 1)))

auc_proposed = roc_auc_score(benchmark_labels, proposed_hmm_scores)

results = {
    'B0_random':          float(auc_b0),
    'B1_position':        float(auc_b1),
    'B2_clip_if':         float(auc_b2),
    'B3_dino_if':         float(auc_b3),
    'B4_sbert':           float(auc_b4) if auc_b4 is not None else None,
    'proposed_hmm':       float(auc_proposed),
}

print('\n=== 베이스라인 비교 (합성 벤치마크) ===')
for name, auc in results.items():
    if auc is None: print(f'  {name}: skip'); continue
    bar = '█' * int(auc * 40)
    print(f'  {name:<22} AUC={auc:.4f}  {bar}')

with open(f'{MODELS_DIR}/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# 비교 시각화
valid_results = {k: v for k, v in results.items() if v is not None}
names  = list(valid_results.keys())
values = list(valid_results.values())
colors = ['steelblue' if n.startswith('proposed') else 'lightcoral' if n.startswith('B') else 'gray' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 바 차트
axes[0].barh(names, values, color=colors, edgecolor='white')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Random (AUC=0.5)')
axes[0].set_xlabel('AUC')
axes[0].set_xlim(0, 1)
axes[0].set_title('베이스라인 비교 — 합성 구조 이상 벤치마크')
axes[0].legend()

# ROC 커브 비교
score_map = {
    'B0_random': random_scores, 'B1_position': position_scores,
    'B2_clip_if': bench_scores_clip, 'B3_dino_if': bench_scores_dino,
    'proposed_hmm': proposed_hmm_scores,
}
for name, scores in score_map.items():
    if scores is None: continue
    fpr, tpr, _ = roc_curve(benchmark_labels, scores)
    auc = valid_results.get(name, 0)
    ls = '-' if name.startswith('proposed') else '--'
    axes[1].plot(fpr, tpr, label=f'{name} ({auc:.3f})', linestyle=ls)
axes[1].plot([0,1],[0,1],'k:', alpha=0.5)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve 비교')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/baseline_comparison.png', dpi=120)
plt.show()
print('저장: baseline_comparison.png')

In [ ]:
print('=== Notebook 7 완료 ===')
print(f'베이스라인 결과: {MODELS_DIR}/baseline_results.json')
print(f'비교 플롯: {MODELS_DIR}/baseline_comparison.png')
print()
print('핵심 해석:')
if auc_b1 > auc_proposed:
    print('  ⚠ B1(position)이 제안 방법보다 높음 — 덱 길이 편차가 주 신호임')
if auc_b3 > auc_proposed:
    print('  ⚠ B3(DINOv2)가 제안 방법보다 높음 — 학습 없는 방법이 더 나음')
if auc_proposed > max(float(auc_b2), float(auc_b3)):
    print('  ✓ HMM이 CLIP/DINOv2 IF보다 높음 — 시퀀스 구조 모델링 기여 입증')
print('\nNotebook 8 (Sensitivity Analysis)으로 이동하세요.')